# Prueba rápida: Qwen3-1.7B con su tool calling nativo

**Pregunta:** ¿el modelo usa mejor la herramienta si se la pasamos con su mecanismo nativo (`tools=` en el chat template) en vez de describirla en el texto del prompt?

**Qué cambia respecto de la línea base:** solo cómo se presenta la herramienta.
- Se quita del prompt la descripción de `get_inventory` y el ejemplo de `<tool_call>`. Queda la frase "No conoces el stock..." para que el modelo sepa que debe consultar.
- La herramienta se pasa como esquema JSON con `tools=`. El chat template de Qwen arma con eso el bloque de sistema con que fue entrenado.
- Los resultados vuelven como mensajes de rol `tool`, seguidos de un mensaje `user` con las mismas instrucciones de respuesta que en la línea base.

Todo lo demás es igual: las reglas, la solicitud, thinking activado, el muestreo (temperatura 0,7, `top_p` 0,95, `top_k` 50), 4.096 tokens por turno y máximo 2 rondas de llamadas.

**Cómo se califica:** la conversación se traduce al formato del harness (el mismo texto de llamada que escribió el modelo, más el mensaje de resultados que arma `render_tool_reply`) y se evalúa con `evaluate_trajectory`. Es el mismo verificador de la línea base, sin cambios.

**Tamaño:** 18 instancias de `eval` en la condición tools (2 por familia × dificultad), 1 muestra cada una. Es una prueba de forma, no una medición: con 18 instancias solo una diferencia grande dice algo. La métrica principal es **cuántas veces usa la etiqueta `<tool_call>`**. En la línea base, en estas mismas 18 instancias, la usó en 5 y acertó 6.

**Requisitos:** GPU (T4 o A100) y el zip del proyecto en `MyDrive/inventory_harness/inventory_harness_colab.zip`, igual que en `docs/guia_colab.md`. Con A100 tarda unos 10–15 minutos; con T4, unos 30–45.

## 1. GPU y configuración

In [ ]:
import torch

assert torch.cuda.is_available(), "Activa la GPU: Entorno de ejecución > Cambiar tipo de entorno"
GPU = torch.cuda.get_device_name(0)
MEMORIA = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {GPU} ({MEMORIA:.1f} GiB)")

MODELO = "Qwen/Qwen3-1.7B"
# T4 no tiene bfloat16 rapido; la linea base uso float16 en T4.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
# El cache de 18 conversaciones de ~6.000 tokens no cabe en los 15 GiB de una T4.
LOTE = 18 if MEMORIA > 30 else 6
N_POR_CELDA = 2          # instancias por familia x dificultad (18 en total)
MAX_NEW_TOKENS = 4096    # por turno, con el <think> incluido, como en la linea base
SEMILLA = 2026
print("dtype:", DTYPE, "| lote:", LOTE)

## 2. Drive y código del proyecto

In [ ]:
import os, shutil, sys, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/inventory_harness")
ZIP_CODIGO = BASE / "inventory_harness_colab.zip"
assert ZIP_CODIGO.exists(), f"Sube inventory_harness_colab.zip a {BASE}"

CODIGO = Path("/content/inventory_harness")
shutil.rmtree(CODIGO, ignore_errors=True)
with zipfile.ZipFile(ZIP_CODIGO) as archivo:
    archivo.extractall(CODIGO)
sys.path.insert(0, str(CODIGO / "src"))
os.chdir(CODIGO)
SALIDA = BASE / "results" / "tools_nativo"
SALIDA.mkdir(parents=True, exist_ok=True)
print("Proyecto en", CODIGO, "| resultados en", SALIDA)

In [ ]:
!pip install -q "transformers>=4.51" accelerate

## 3. Instancias

Las mismas de la línea base: `eval`, 10 por celda, semilla 2026. La huella debe coincidir con la del `manifest.json` de la línea base; si no coincide, el zip tiene otra versión del generador y la comparación no vale.

In [ ]:
import hashlib, json
from collections import defaultdict
from inventory_harness.generator import generate_dataset
from inventory_harness.schema import SPLITS

HUELLA_LINEA_BASE = "efb77852b9c86edc5c478be0de30fa11515d288e272a6cf09f624c3bd0453e5b"

config = {"n_per_cell": {nombre: 10 for nombre in SPLITS}, "base_seed": 2026}
eval_ = [i for i in generate_dataset(config) if i.split == "eval"]
digest = hashlib.sha256()
for instancia in sorted(eval_, key=lambda item: item.id):
    digest.update(json.dumps(instancia.to_dict(), sort_keys=True).encode("utf-8"))
assert digest.hexdigest() == HUELLA_LINEA_BASE, "El dataset no coincide con el de la linea base"

sufijos = tuple(f"_{n:04d}" for n in range(N_POR_CELDA))
INSTANCIAS = sorted((i for i in eval_ if i.id.endswith(sufijos)), key=lambda i: i.id)
print(len(INSTANCIAS), "instancias:", [i.id for i in INSTANCIAS])

## 4. Prompt y esquema de la herramienta

El prompt es el de la línea base (`render_prompt(instancia, "tools")`) con un solo cambio: la sección `Herramienta:` se reemplaza por una línea que dice que el stock se obtiene con `get_inventory`. La descripción pasa al esquema.

In [ ]:
from inventory_harness.prompts import _TOOL_SECTION, render_prompt

assert _TOOL_SECTION.startswith("Herramienta:")
AVISO = "No conoces el stock y no puedes suponerlo: los valores solo se obtienen con la herramienta get_inventory."

TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_inventory",
        "description": ("Devuelve physical_stock y reserved_stock de un producto. "
                        "Sin location_id entrega todas sus sucursales."),
        "parameters": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string", "description": "ID del producto del catálogo, por ejemplo P01"},
                "location_id": {"type": "string", "description": "ID de la sucursal del catálogo, por ejemplo S1. Opcional"},
            },
            "required": ["product_id"],
        },
    },
}]


def prompt_nativo(instancia):
    original = render_prompt(instancia, "tools")
    assert _TOOL_SECTION in original
    return original.replace(_TOOL_SECTION, AVISO)


print(prompt_nativo(INSTANCIAS[0]))

## 5. Modelo

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODELO)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=DTYPE, device_map="cuda")
model.eval()

# Si <tool_call> fuera un token especial, decodificar con skip_special_tokens=True lo
# borraria. Aqui se decodifica sin saltarlos, pero conviene saberlo para leer la linea base.
especiales = set(tok.all_special_tokens)
for etiqueta in ("<tool_call>", "</tool_call>", "<think>"):
    print(etiqueta, "id", tok.convert_tokens_to_ids(etiqueta), "| especial:", etiqueta in especiales)

# Asi ve el modelo el prompt: el bloque de sistema con <tools> lo arma el template.
ejemplo = tok.apply_chat_template([{"role": "user", "content": prompt_nativo(INSTANCIAS[0])}],
                                  tools=TOOLS, tokenize=False, add_generation_prompt=True,
                                  enable_thinking=True)
print(ejemplo[:1500])

## 6. Generación

Un loop corto: genera, lee las llamadas con el mismo parser del harness, las ejecuta y devuelve los resultados como mensajes `tool`. Las conversaciones se generan en lotes.

In [ ]:
import re, time
from inventory_harness.backends import cut_at_stop
from inventory_harness.prompts import render_tool_reply
from inventory_harness.rollout import restore_answer_close
from inventory_harness.tools import (MAX_TOOL_ROUNDS, execute_tool_call, parse_tool_calls,
                                     render_observations, truncate_at_tool_result, visible_start)

PARADAS = ("</answer>", "<tool_result>", "<tool_response>")
FIN = ("<|im_end|>", "<|endoftext|>")
ETIQUETA = re.compile(r"<\s*tool_call\s*>", re.IGNORECASE)


def generar(conversaciones):
    textos = [tok.apply_chat_template(c, tools=TOOLS, tokenize=False, add_generation_prompt=True,
                                      enable_thinking=True) for c in conversaciones]
    entrada = tok(textos, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        salida = model.generate(**entrada, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                temperature=0.7, top_p=0.95, top_k=50,
                                stop_strings=list(PARADAS), tokenizer=tok,
                                pad_token_id=tok.pad_token_id)
    resultados = []
    for fila in salida[:, entrada["input_ids"].shape[1]:]:
        texto = tok.decode(fila, skip_special_tokens=False)
        cortado = len(texto)
        for fin in FIN:
            if fin in texto:
                cortado = min(cortado, texto.index(fin))
        texto = texto[:cortado]
        texto, parada = cut_at_stop(texto, PARADAS)
        if parada == "<tool_response>":
            texto = texto[: -len(parada)].rstrip()  # el modelo invento un resultado
        texto = restore_answer_close(texto, "answer" if parada == "</answer>" else None)
        resultados.append(truncate_at_tool_result(texto))
    return resultados


estado = [{"inst": i, "nativo": [{"role": "user", "content": prompt_nativo(i)}],
           "canon": [], "rondas": 0, "listo": False, "etiquetas": []} for i in INSTANCIAS]

torch.manual_seed(SEMILLA)
inicio = time.time()
for turno in range(MAX_TOOL_ROUNDS + 1):
    activos = [e for e in estado if not e["listo"]]
    if not activos:
        break
    for desde in range(0, len(activos), LOTE):
        lote = activos[desde:desde + LOTE]
        for e, texto in zip(lote, generar([e["nativo"] for e in lote])):
            e["etiquetas"].append(bool(ETIQUETA.search(texto, visible_start(texto))))
            e["canon"].append({"role": "assistant", "content": texto})
            llamadas = parse_tool_calls(texto)
            if not llamadas or e["rondas"] >= MAX_TOOL_ROUNDS:
                e["listo"] = True  # respuesta final, o una tercera ronda que no se ejecuta
                continue
            e["rondas"] += 1
            resultados = [execute_tool_call(e["inst"], llamada) for llamada in llamadas]
            respuesta = render_tool_reply(e["inst"], resultados, e["rondas"])
            observaciones = render_observations(resultados)
            assert respuesta.startswith(observaciones)
            e["canon"].append({"role": "user", "content": respuesta})
            e["nativo"].append({"role": "assistant", "content": texto})
            e["nativo"] += [{"role": "tool", "content": json.dumps(r, sort_keys=True, ensure_ascii=False)}
                            for r in resultados]
            e["nativo"].append({"role": "user", "content": respuesta[len(observaciones):].lstrip("\n")})
    print(f"turno {turno + 1}: {len(activos)} conversaciones, {time.time() - inicio:.0f} s acumulados")

## 7. Resultados frente a la línea base

La línea base es la muestra 0 de cada instancia en `results/kaggle_qwen17b/linea_base_qwen17b`, con el verificador actual.

In [ ]:
from collections import Counter
from inventory_harness.reward import evaluate_trajectory

LINEA_BASE = {  # muestra 0, condicion tools: acierto y uso de <tool_call> en el primer mensaje
    "eval_availability_easy_0000": (1, False), "eval_availability_easy_0001": (1, False),
    "eval_availability_medium_0000": (1, False), "eval_availability_medium_0001": (0, False),
    "eval_availability_hard_0000": (0, True), "eval_availability_hard_0001": (0, True),
    "eval_selection_easy_0000": (1, False), "eval_selection_easy_0001": (0, True),
    "eval_selection_medium_0000": (0, False), "eval_selection_medium_0001": (1, True),
    "eval_selection_hard_0000": (0, False), "eval_selection_hard_0001": (0, False),
    "eval_composition_easy_0000": (0, False), "eval_composition_easy_0001": (1, False),
    "eval_composition_medium_0000": (0, False), "eval_composition_medium_0001": (0, False),
    "eval_composition_hard_0000": (0, True), "eval_composition_hard_0001": (0, False),
}

filas = []
for e in estado:
    v = evaluate_trajectory(e["inst"], "tools", e["canon"])
    base_ok, base_tag = LINEA_BASE[e["inst"].id]
    filas.append({"id": e["inst"].id, "recompensa": v.reward, "error": v.error_type,
                  "etiqueta_1er_msg": e["etiquetas"][0], "llamadas_validas": v.valid_tool_call,
                  "datos_completos": v.retrieved_required_data, "rondas": e["rondas"],
                  "base_recompensa": base_ok, "base_etiqueta": base_tag})

print(f"{'instancia':32} {'nativo':>7} {'base':>5} {'tag':>4} {'tag base':>8}  error")
for f in filas:
    print(f"{f['id']:32} {f['recompensa']:7.0f} {f['base_recompensa']:5d} "
          f"{'sí' if f['etiqueta_1er_msg'] else 'no':>4} {'sí' if f['base_etiqueta'] else 'no':>8}  {f['error']}")

n = len(filas)
print()
print(f"Aciertos:                   nativo {sum(f['recompensa'] for f in filas):.0f}/{n} | base {sum(f['base_recompensa'] for f in filas)}/{n}")
print(f"Usa <tool_call> (1er msg):  nativo {sum(f['etiqueta_1er_msg'] for f in filas)}/{n} | base {sum(f['base_etiqueta'] for f in filas)}/{n}")
print(f"Llamadas todas válidas:     nativo {sum(bool(f['llamadas_validas']) for f in filas)}/{n}")
print(f"Reunió los datos necesarios: nativo {sum(bool(f['datos_completos']) for f in filas)}/{n}")
print("Errores (nativo):", Counter(f["error"] for f in filas))

## 8. Ver una conversación

In [ ]:
ID = INSTANCIAS[0].id   # cambia el ID para ver otra
e = next(e for e in estado if e["inst"].id == ID)
for mensaje in e["nativo"][1:] + [e["canon"][-1]]:
    contenido = re.sub(r"(?s)<think>.*?</think>", "<think>[...]</think>", mensaje["content"])
    print(f"--- {mensaje['role']}\n{contenido[:1500]}\n")

## 9. Guardar

In [ ]:
archivo = SALIDA / f"tools_nativo_{time.strftime('%Y%m%d_%H%M')}.jsonl"
with archivo.open("w", encoding="utf-8") as salida:
    for e, f in zip(estado, filas):
        salida.write(json.dumps({**f, "mensajes_nativos": e["nativo"], "mensajes_harness": e["canon"]},
                                ensure_ascii=False) + "\n")
print("Guardado en", archivo)